In [0]:
%run ./22-tumbling-time-window

In [0]:
class TradeSummaryTestSuite():
    def __init__(self):
        import os
        self.base_data_dir = os.path.abspath("..")
        self.checkpoint_dir = ("/Volumes/spark_stream/default/streaming_data/checkpoints")

    def cleanTests(self):
        print(f"Starting Cleanup...", end='')
        spark.sql("drop table if exists spark_stream.default.kafka_bz")
        spark.sql("drop table if exists spark_stream.default.trade_summary")

        spark.sql(f"CREATE TABLE if not EXISTS spark_stream.default.kafka_bz(key STRING, value STRING)")

        dbutils.fs.rm(f"{self.checkpoint_dir}/trade_summary", True)
        print("Done")

    def assertTradeSummary(self, start, end, expected_buy, expected_sell):
        print(f"\tStarting Trade Summary validation...", end='')
        result = (spark.sql(f'''select TotalBuy
                                , TotalSell 
                                from spark_stream.default.trade_summary 
                                    where date_format(start, 'yyyy-MM-dd HH:mm:ss') = '{start}' 
                                    and date_format(end, 'yyyy-MM-dd HH:mm:ss')='{end}'
                            ''')
                            .collect()
                )
        actual_buy = result[0][0]
        actual_sell = result[0][1]
        assert expected_buy == actual_buy, f"Test failed! actual buy is {actual_buy}"
        assert expected_sell == actual_sell, f"Test failed! actual sell is {actual_sell}"
        print("Done")

    def runTests(self):
        self.cleanTests()

        stream = TradeSummary()      

        print("\nTesting first two events...") 
        spark.sql("""INSERT INTO spark_stream.default.kafka_bz VALUES
                  ('2019-02-05', '{"CreatedTime": "2019-02-05 10:05:00", "Type": "BUY", "Amount": 500, "BrokerCode": "ABX"}'),
                  ('2019-02-05', '{"CreatedTime": "2019-02-05 10:12:00", "Type": "BUY", "Amount": 300, "BrokerCode": "ABX"}')
            """)
        sQuery = stream.process() 
        sQuery.awaitTermination()
        self.assertTradeSummary('2019-02-05 10:00:00', '2019-02-05 10:15:00', 800, 0)

        print("\nTesting third and fourth events...") 
        spark.sql("""INSERT INTO spark_stream.default.kafka_bz VALUES
                  ('2019-02-05', '{"CreatedTime": "2019-02-05 10:20:00", "Type": "BUY", "Amount": 600, "BrokerCode": "ABX"}'),
                  ('2019-02-05', '{"CreatedTime": "2019-02-05 10:40:00", "Type": "BUY", "Amount": 900, "BrokerCode": "ABX"}')
            """)
        sQuery = stream.process() 
        sQuery.awaitTermination()        
        self.assertTradeSummary('2019-02-05 10:15:00', '2019-02-05 10:30:00', 600, 0)
        self.assertTradeSummary('2019-02-05 10:30:00', '2019-02-05 10:45:00', 900, 0)

        print("\nTesting late event...") 
        spark.sql("""INSERT INTO spark_stream.default.kafka_bz VALUES
                    ('2019-02-05', '{"CreatedTime": "2019-02-05 10:48:00", "Type": "SELL", "Amount": 500, "BrokerCode": "ABX"}'),
                    ('2019-02-05', '{"CreatedTime": "2019-02-05 10:25:00", "Type": "SELL", "Amount": 400, "BrokerCode": "ABX"}')
            """)
        sQuery = stream.process() 
        sQuery.awaitTermination()        
        self.assertTradeSummary('2019-02-05 10:45:00', '2019-02-05 11:00:00', 0, 500)
        self.assertTradeSummary('2019-02-05 10:15:00', '2019-02-05 10:30:00', 600, 400)

        print("Validation passed.\n")        

In [0]:
ts = TradeSummaryTestSuite()
ts.runTests()